In [ ]:
import matplotlib.pyplot as plt
import re
import gzip
import shutil
import pandas as pd
import scipy.sparse
from multiprocessing import Pool
import tarfile
import seaborn as sns
import scanpy.external as sce 
import harmonypy as hm
import anndata as ad
import omicverse as ov
import scvi
import scib
import annoy
import gc
import numpy as np
import random
import scanpy as sc
import os

In [ ]:
sc.settings.verbosity = 1 
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4), facecolor='white')
ov.style(font_path='arial')

In [ ]:
os.chdir("")
os.getcwd()

In [ ]:
def save_fig(
    fig_name,
    save_dir="",
    fmt="pdf",
    dpi=300,
):
    save_fig.counter = getattr(save_fig, "counter", 1)
    filename = f"{save_fig.counter:02d}_{fig_name}.{fmt}"
    plt.savefig(
        os.path.join(save_dir, filename),
        format=fmt,
        dpi=dpi,
        bbox_inches="tight",
    )
    save_fig.counter += 1

In [ ]:
adata_mono2 = ov.read("")

In [ ]:
adata_mono2

In [ ]:
adata_allo = adata_mono2[adata_mono2.obs["orig.ident"] == "OMIX012347"].copy()
adata_xeno = adata_mono2[adata_mono2.obs["orig.ident"] == "Xen_17390399"].copy()

In [ ]:
Traj = ov.single.TrajInfer(
    adata_xeno,
    basis="X_pca",
    groupby="mono_subtype",
    use_rep="X_pca",
    n_comps=30,
)

In [ ]:
Traj.set_origin_cells("CD14+ Mono")

In [ ]:
import pcurvepy2

In [ ]:
Traj.inference(method='slingshot', num_epochs=1)

In [ ]:
ov.pl.embedding(
    adata_xeno,
    basis="X_umap",
    color=["mono_subtype", "slingshot_pseudotime"],
    frameon="small",
    cmap="Reds",
)

Supplementary Fig 3B

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ov.pl.embedding(
    adata_xeno,
    basis="X_umap",
    color="mono_subtype",
    ax=ax,
    show=False,
    size=30,
)

ov.pl.trajectory_overlay(
    adata_xeno,
    ax=ax,
    method="slingshot",
    model=Traj.slingshot,
)
plt.show()

In [ ]:
sc.pp.neighbors(adata_xeno, use_rep='X_pca_harmony')

ov.utils.cal_paga(
    adata_xeno,
    use_time_prior='slingshot_pseudotime',
    vkey='paga',
    groups='mono_subtype'
)


In [ ]:
ov.utils.plot_paga(
    adata_xeno,
    basis='umap',  
    size=30,
    alpha=0.1,
    title='PAGA Slingshot Directed Graph',
    min_edge_width=2,
    node_size_scale=1.5,
    show=False,
    legend_loc=False
)
plt.show()

Figure 4A

In [ ]:
from scipy.ndimage import gaussian_filter1d

def fix_and_smooth_curve(raw_pts, sigma=11.0, n_bins=100):
    diff = np.diff(raw_pts, axis=0)
    dist = np.sqrt((diff ** 2).sum(axis=1))
    valid = np.insert(dist > 1e-4, 0, True)
    pts = raw_pts[valid]

    diff = np.diff(pts, axis=0)
    cum_dist = np.insert(np.cumsum(np.sqrt((diff ** 2).sum(axis=1))), 0, 0.0)

    t_uniform = np.linspace(0, cum_dist[-1], n_bins)
    x_interp = np.interp(t_uniform, cum_dist, pts[:, 0])
    y_interp = np.interp(t_uniform, cum_dist, pts[:, 1])

    x_smooth = gaussian_filter1d(x_interp, sigma=sigma, mode="nearest")
    y_smooth = gaussian_filter1d(y_interp, sigma=sigma, mode="nearest")

    x_smooth[-1] = pts[-1, 0]
    y_smooth[-1] = pts[-1, 1]
    return x_smooth, y_smooth

fig = plt.figure(figsize=(9, 6.5), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[6.5, 2.5], wspace=0.05)

ax = fig.add_subplot(gs[0])
ax_leg = fig.add_subplot(gs[1])
ax_leg.axis("off")

ax.set_aspect("equal", adjustable="box")

ov.pl.embedding(
    adata_xeno,
    basis="X_umap",
    color="mono_subtype",
    ax=ax,
    show=False,
    size=12,
    alpha=0.3,
    frameon="small",
)

for name, info in lineages.items():
    raw_pts = info["pts"]
    sig = 11.0 if "PF4" in name else 4.0
    sx, sy = fix_and_smooth_curve(raw_pts, sigma=sig, n_bins=100)

    ax.plot(
        sx, sy, color="white", linewidth=6.0, zorder=10, solid_capstyle="round"
    )
    ax.plot(
        sx,
        sy,
        color=info["color"],
        linewidth=3.2,
        label=name,
        zorder=11,
        solid_capstyle="round",
    )

    ax.annotate(
        "",
        xy=(sx[-1], sy[-1]),
        xytext=(sx[-6], sy[-6]),
        arrowprops=dict(
            arrowstyle="-|>",
            color=info["color"],
            lw=3.2,
            mutation_scale=18,
        ),
        zorder=12,
    )

ax.set_title(
    "Slingshot Trajectories: Monocyte Fate Commitment",
    fontweight="bold",
    pad=12,
    fontsize=13,
)

handles, labels = ax.get_legend_handles_labels()
if ax.get_legend() is not None:
    ax.get_legend().remove()

ax_leg.legend(
    handles=handles,
    labels=labels,
    loc="center left",
    frameon=False,
    fontsize=10,
    labelspacing=0.8,
)

save_fig("Slingshot_UMAP_Trajectories")
plt.show()

Fgiure 4B

In [ ]:
genes = [
    "CD14",
    "VCAN",
    "S100A12",
    "HLA-DRA",
    "FCGR3A",
    "PF4",
    "PPBP",
    "TUBB1",
    "CLU",
]
available_var = (
    adata_xeno.raw.var_names
    if adata_xeno.raw is not None
    else adata_xeno.var_names
)
final_trend_genes = [g for g in genes if g in available_var]

slingshot_global_dyn = ov.single.dynamic_features(
    adata_xeno,
    genes=final_trend_genes,
    pseudotime="slingshot_pseudotime",
    use_raw=adata_xeno.raw is not None,
    distribution="normal",
    link="identity",
    n_splines=6,
    store_raw=True,
    raw_obs_keys=["mono_subtype"],
)

fig = ov.pl.dynamic_trends(
    slingshot_global_dyn,
    genes=final_trend_genes,
    add_point=True,
    point_color_by="mono_subtype",
    figsize=(3.8, 2.8),
    legend_loc="right margin",
    legend_fontsize=8,
    return_fig=True,
)

save_fig("Slingshot_UMAP_Dynamic_Genes")
plt.show()

In [ ]:
batch_col = "batch"
subtype_col = "mono_subtype"

np.random.seed(42)
sampled_barcodes = []
max_cells_per_stratum = 150

for (b, s), group_df in adata_xeno.obs.groupby(
    [batch_col, subtype_col], observed=True
):
    cell_ids = group_df.index.tolist()
    if len(cell_ids) > max_cells_per_stratum:
        sampled_barcodes.extend(
            np.random.choice(
                cell_ids, size=max_cells_per_stratum, replace=False
            )
        )
    else:
        sampled_barcodes.extend(cell_ids)

adata_scenic = adata_xeno[sampled_barcodes].copy()

In [ ]:
adata_scenic.X = adata_scenic.layers["lognorm"].copy()

sc.pp.highly_variable_genes(
    adata_scenic, n_top_genes=2000, flavor="seurat"
)
adata_scenic = adata_scenic[:, adata_scenic.var["highly_variable"]].copy()

In [ ]:
scenic_dir = "/data/zdh/ZDH_Spatial/Python包/scenic/human_hg38"

scenic_obj = ov.single.SCENIC(
    adata=adata_scenic,
    species="human",
    db_glob=f"{scenic_dir}/*.genes_vs_motifs.rankings.feather",
    motif_path=f"{scenic_dir}/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl",
    n_jobs=8,
)


In [ ]:
edgelist = scenic_obj.cal_grn(layer="counts")


In [ ]:
regulon_ad = scenic_obj.cal_regulons(
    rho_mask_dropouts=True,
    thresholds=(0.75, 0.9),
    top_n_targets=(50,),
    top_n_regulators=(5, 10, 50),
)


In [ ]:
regulon_ad.obs["slingshot_pseudotime"] = adata_xeno.obs.loc[
    regulon_ad.obs_names, "slingshot_pseudotime"
]
regulon_ad.obs["mono_subtype"] = adata_xeno.obs.loc[
    regulon_ad.obs_names, "mono_subtype"
]
regulon_ad.obsm["X_umap"] = adata_xeno.obsm["X_umap"][
    adata_xeno.obs_names.isin(regulon_ad.obs_names)
]


In [ ]:
import math
from adjustText import adjust_text
from omicverse.external.pyscenic.plotting import plot_rss
from omicverse.external.pyscenic.rss import regulon_specificity_scores

cell_subtypes = scenic_obj.adata.obs["mono_subtype"]

rss = regulon_specificity_scores(
    scenic_obj.auc_mtx, 
    cell_subtypes, 
)


Figure 4C

Supplementary Fig3A

In [ ]:


target_terminal_cluster = "PF4+PPBP+ Mono"
cats = sorted(list(set(cell_subtypes.dropna())))
n_cats = len(cats)
ncols = 3
nrows = math.ceil(n_cats / ncols)

fig = plt.figure(figsize=(14, 4 * nrows))

for num, c in enumerate(cats, 1):
    x = rss.T[c].dropna()
    ax = fig.add_subplot(nrows, ncols, num)
    plot_rss(rss, c, top_n=5, max_n=None, ax=ax)
    ax.set_ylim(
        x.min() - (x.max() - x.min()) * 0.05,
        x.max() + (x.max() - x.min()) * 0.05,
    )
    for t in ax.texts:
        t.set_fontsize(9)
    ax.set_title(c, fontsize=11, fontweight="bold")
    ax.set_ylabel("")
    ax.set_xlabel("")
    adjust_text(
        ax.texts,
        autoalign="xy",
        ha="right",
        va="bottom",
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        precision=0.001,
    )

fig.text(0.5, 0.02, "Regulons", ha="center", va="center", size=13)
fig.text(
    0.01,
    0.5,
    "Regulon Specificity Score (RSS)",
    ha="center",
    va="center",
    rotation="vertical",
    size=13,
)
plt.tight_layout(rect=[0.02, 0.03, 1, 0.98])
plt.show()

In [ ]:
auc_df = scenic_obj.auc_mtx.loc[adata_scenic.obs_names].copy()

adata_auc = ov.AnnData(
    X=auc_df.values,
    obs=adata_scenic.obs.copy(),
    var=pd.DataFrame(index=auc_df.columns),
)

adata_auc.obs["slingshot_pseudotime"] = adata_xeno.obs.loc[
    adata_auc.obs_names, "slingshot_pseudotime"
]
adata_auc.obs["mono_subtype"] = adata_xeno.obs.loc[
    adata_auc.obs_names, "mono_subtype"
]

regulons_focus = [
    "JUN(+)",
    "FOS(+)",
    "FOSB(+)",
    "RFX2(+)",
    "ZEB1(+)",
    "MITF(+)",
    "NFIA(+)",
    "HIVEP1(+)",
    "PBX3(+)",
]

final_regulons = [r for r in regulons_focus if r in adata_auc.var_names]

regulon_dyn = ov.single.dynamic_features(
    adata_auc,
    genes=final_regulons,
    pseudotime="slingshot_pseudotime",
    distribution="normal",
    link="identity",
    n_splines=6,
    store_raw=True,
    raw_obs_keys=["mono_subtype"],
)

fig = ov.pl.dynamic_trends(
    regulon_dyn,
    genes=final_regulons,
    add_point=True,
    point_color_by="mono_subtype",
    figsize=(3.8, 2.8),
    legend_loc="right margin",
    legend_fontsize=8,
    return_fig=True,
)

plt.savefig("Slingshot_9Regulons_Dynamic_Trends.pdf", dpi=300, bbox_inches="tight")
plt.savefig("Slingshot_9Regulons_Dynamic_Trends.png", dpi=300, bbox_inches="tight")
plt.show()

Figure 4E & F

Supplementary Fig3C

celloracle

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
base_GRN = co.data.load_human_promoter_base_GRN()

oracle = co.Oracle()
oracle.import_anndata_as_raw_count(
    adata=adata_xeno,
    cluster_column_name="mono_subtype",
    embedding_name="X_umap",
)

# 注入基线调控先验
oracle.import_TF_data(TF_info_matrix=base_GRN)

In [ ]:
oracle.perform_PCA()
k_val = min(30, max(15, int(0.02 * oracle.adata.n_obs)))
oracle.knn_imputation(
    n_pca_dims=30, k=k_val, balanced=True, b_sight=k_val * 8, b_maxl=k_val * 4
)

links = oracle.get_links(
    cluster_name_for_GRN_unit="mono_subtype", alpha=10, verbose_level=0
)
links.filter_links(p=0.001)

oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)

In [ ]:
from celloracle.applications import Oracle_development_module
from matplotlib.quiver import Quiver

SAVE_DIR = ""
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["savefig.dpi"] = 450

SIM_SCALE = 5.0
DEV_SCALE = 40.0
MAX_ARROW_FRACTION = 0.08


def clean_celloracle_figure(fig, dev):
  xy = np.asarray(dev.embedding)
  xy_min = np.nanmin(xy, axis=0)
  xy_max = np.nanmax(xy, axis=0)
  margin = (xy_max - xy_min) * 0.05
  clipped_counts = []

  for i, ax in enumerate(fig.axes[:6]):
    if i < 4:
      ax.set_xlim(xy_min[0] - margin[0], xy_max[0] + margin[0])
      ax.set_ylim(xy_min[1] - margin[1], xy_max[1] + margin[1])
      ax.set_autoscale_on(False)

    for artist in ax.collections:
      if isinstance(artist, Quiver):
        u = np.asarray(artist.U)
        v = np.asarray(artist.V)
        magnitude = np.hypot(u, v)
        max_magnitude = MAX_ARROW_FRACTION * artist.scale
        factor = np.minimum(
            1.0, max_magnitude / np.maximum(magnitude, 1e-12)
        )
        artist.set_UVC(u * factor, v * factor)
        clipped_counts.append(int(np.sum(magnitude > max_magnitude)))

      if i < 4:
        artist.set_clip_on(True)
        artist.set_clip_path(ax.patch)

      artist.set_rasterized(True)

  return sum(clipped_counts)


candidate_tfs = [
    "JUN",
    "FOSB",
    "FOS",
    "RFX2",
    "ZEB1",
    "MITF",
    "NFIA",
    "HIVEP1",
    "PBX3",
]
active_tfs = set(oracle.active_regulatory_genes)
valid_tfs = [tf for tf in candidate_tfs if tf in active_tfs]
cluster_mask = (oracle.adata.obs["mono_subtype"] == "PF4+PPBP+ Mono").to_numpy()

scores = {}

for tf in valid_tfs:
  print(f"\n正在模拟 {tf} KO ...")

  oracle.simulate_shift(
      perturb_condition={tf: 0.0}, n_propagation=3, ignore_warning=True
  )
  oracle.estimate_transition_prob(
      n_neighbors=200, knn_random=True, sampled_fraction=1
  )
  oracle.calculate_embedding_shift(sigma_corr=0.05)

  scores[tf] = np.linalg.norm(
      oracle.delta_embedding[cluster_mask], axis=1
  ).mean()

  dev = Oracle_development_module(
      oracle_object=oracle, gradient_object=gradient
  )
  dev.calculate_inner_product()
  dev.calculate_digitized_ip(n_bins=10)

  fig = dev.visualize_development_module_layout_0(
      s=5,
      scale_for_simulation=SIM_SCALE,
      scale_for_pseudotime=DEV_SCALE,
      s_grid=35,
      vm=0.02,
      return_fig=True,
  )

  n_clipped = clean_celloracle_figure(fig, dev)
  fig.suptitle(f"{tf} KO - CellOracle Perturbation", fontsize=18)
  fig.tight_layout(rect=[0, 0, 1, 0.96])

  plt.figure(fig.number)
  save_fig(f"CellOracle_{tf}_KO_Development_Flow")
  plt.show()
  plt.close(fig)

  print(f"✅ {tf} 完成 | 显示长度受限的箭头数: {n_clipped}")

ranking = pd.Series(
    {tf: scores.get(tf, np.nan) for tf in candidate_tfs},
    name="Mean_UMAP_Displacement",
).sort_values(ascending=False)

display(ranking)

ranking.to_csv(os.path.join(SAVE_DIR, "CellOracle_TF_KO_Ranking.csv"))

Supplementary Fig3D

In [ ]:
SAVE_DIR = ""
os.makedirs(SAVE_DIR, exist_ok=True)

platelet_genes = [
    "PF4",
    "PPBP",
    "GP9",
    "GP1BA",
    "GP1BB",
    "TUBB1",
    "RGS18",
    "ITGA2B",
    "ITGB3",
    "TREML1",
    "MPIG6B",
    "NRGN",
    "SDPR",
    "CLU",
]
inflammatory_genes = [
    "S100A8",
    "S100A9",
    "S100A12",
    "IL1A",
    "IL1B",
    "TNF",
    "CXCL8",
    "CCL2",
]


In [ ]:

genes = [
    g for g in platelet_genes + inflammatory_genes if g in oracle.adata.var_names
]
baseline = sc.get.obs_df(oracle.adata, keys=genes, layer="imputed_count")
sd = baseline.std(ddof=0)
genes = sd[sd > 0].index.tolist()
baseline = baseline[genes]
sd = sd[genes]

platelet_used = [g for g in platelet_genes if g in genes]
inflammatory_used = [g for g in inflammatory_genes if g in genes]
subtypes = oracle.adata.obs["mono_subtype"].astype(str)
subtype_order = list(pd.unique(subtypes))

changes = {}
for tf in valid_tfs:
  oracle.simulate_shift(
      perturb_condition={tf: 0.0}, n_propagation=3, ignore_warning=True
  )
  simulated = sc.get.obs_df(oracle.adata, keys=genes, layer="simulated_count")
  delta = (simulated - baseline) / sd
  changes[tf] = delta.groupby(subtypes).mean().reindex(subtype_order)

gene_change = pd.concat(changes, names=["TF", "Subtype"])
gene_change.to_csv(
    os.path.join(SAVE_DIR, "CellOracle_9TF_AllSubtypes_GeneChanges.csv")
)

pf_change = gene_change.xs("PF4+PPBP+ Mono", level="Subtype").reindex(valid_tfs)[
    genes
]
v_gene = np.nanmax(np.abs(pf_change.values))

fig, ax = plt.subplots(figsize=(12, 4.5), dpi=300)
im = ax.imshow(pf_change, cmap="RdBu_r", aspect="auto", vmin=-v_gene, vmax=v_gene)
ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=90, fontsize=9.5)
ax.set_yticks(range(len(valid_tfs)))
ax.set_yticklabels(valid_tfs, fontsize=10, fontweight="bold")
ax.axvline(len(platelet_used) - 0.5, color="black", lw=1.2)
ax.set_title("Predicted Gene Expression Changes in PF4+PPBP+ Mono", fontsize=12, pad=10)
fig.colorbar(im, ax=ax, label="Standardized expression shift", shrink=0.85)
plt.tight_layout()
save_fig("CellOracle_9TF_PF4PPBP_Gene_Heatmap")
plt.show()


In [ ]:

module_change = pd.DataFrame({
    "Platelet": gene_change[platelet_used].mean(axis=1),
    "Inflammatory": gene_change[inflammatory_used].mean(axis=1),
})
module_change.to_csv(
    os.path.join(SAVE_DIR, "CellOracle_9TF_AllSubtypes_ModuleChanges.csv")
)
v_mod = np.nanmax(np.abs(module_change.values))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=300, layout="constrained")
for ax, module in zip(axes, ["Platelet", "Inflammatory"]):
  mat = (
      module_change[module]
      .unstack("Subtype")
      .reindex(index=valid_tfs, columns=subtype_order)
  )
  im = ax.imshow(mat, cmap="RdBu_r", aspect="auto", vmin=-v_mod, vmax=v_mod)
  ax.set_xticks(range(len(subtype_order)))
  ax.set_xticklabels(subtype_order, rotation=35, ha="right", fontsize=9.5)
  ax.set_yticks(range(len(valid_tfs)))
  ax.set_yticklabels(valid_tfs, fontsize=10, fontweight="bold")
  ax.set_title(f"{module}-associated program", fontsize=11.5, pad=8)
fig.colorbar(
    im, ax=axes, label="Mean standardized expression shift", shrink=0.8
)
save_fig("CellOracle_9TF_AllSubtypes_Module_Heatmap")
plt.show()

impact = pd.DataFrame({
    "Platelet": gene_change[platelet_used].abs().mean(axis=1),
    "Inflammatory": gene_change[inflammatory_used].abs().mean(axis=1),
})
ranking_genes = impact.xs("PF4+PPBP+ Mono", level="Subtype").sort_values("Platelet")

fig, ax = plt.subplots(figsize=(6.8, 4.6), dpi=300)
ranking_genes.plot.barh(
    ax=ax,
    color=["#4B89A8", "#D98A6C"],
    width=0.72,
    edgecolor="black",
    linewidth=0.8,
)
ax.set_xlabel("Mean absolute standardized expression shift", fontsize=9.5, labelpad=8)
ax.set_ylabel("")
ax.set_title("Predicted TF KO Effects in PF4+PPBP+ Mono", fontsize=11.5, pad=10)
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig("CellOracle_9TF_PF4PPBP_Module_Ranking")
plt.show()

display(ranking_genes.sort_values("Platelet", ascending=False))